In [3]:
import gradio as gr
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import importlib

import crypto_panel
import metals_panel
import macro_price_panel
import Sem_investment_comparison

importlib.reload(crypto_panel)
importlib.reload(metals_panel)
importlib.reload(macro_price_panel)
importlib.reload(Sem_investment_comparison)

from crypto_panel import build_crypto_panel
from metals_panel import build_metals_panel
from macro_price_panel import build_price_panel
from Sem_investment_comparison import make_investment_comparison


# --------------------------------------------------
# Existing dashboard charts
# --------------------------------------------------

def make_charts(data, ticker, period, metal, crypto):

    close = data["Close"].dropna()

    if close.empty:
        raise ValueError("No closing-price data was returned for this ticker.")


    # Chart 3: Interactive five-investment comparison
    comparison_fig = make_investment_comparison(
        data,
        ticker,
        period,
        metal,
        crypto
    )

    return comparison_fig


# --------------------------------------------------
# Error handling and placeholder charts
# --------------------------------------------------

def placeholder_plot(title, message):

    figure, axis = plt.subplots(figsize=(8, 3.2))

    axis.text(
        0.5,
        0.5,
        message,
        ha="center",
        va="center",
        color="#666666"
    )

    axis.set_title(title)
    axis.set_axis_off()
    figure.tight_layout()

    return figure


def update_dashboard(ticker, period, metal, crypto):

    ticker = ticker.strip().upper()

    try:

        if not ticker:
            raise ValueError("Enter a ticker symbol, such as AAPL.")

        data = yf.Ticker(ticker).history(
            period=period,
            auto_adjust=False
        )

        if data.empty:
            raise ValueError(
                "No data was found. Check the ticker or try another period."
            )

        comparison_fig = make_charts(data, ticker, period, metal, crypto)
        
        return (
            f"Loaded {len(data):,} trading days for {ticker}.",
            comparison_fig
        )

    except Exception as error:

        message = str(error)


        return (
            f"Unable to load {ticker or 'ticker'}: {message}", placeholder_plot("Investment comparison", message)

        )


def initial_plot(title):
    return placeholder_plot(
        title,
        "Enter a ticker and select Load charts"
    )


# --------------------------------------------------
# Gradio dashboard
# --------------------------------------------------

demo = gr.Blocks(title="Market charts")

with demo:

    gr.Markdown(
        "# Market charts\n"
        "Explore stock prices, Treasury yields and investment performance."
    )

    with gr.Row():

        ticker_input = gr.Textbox(
            label="Ticker",
            value="AAPL",
            placeholder="AAPL"
        )

        period_input = gr.Dropdown(
            choices=["1mo", "3mo", "6mo", "1y", "2y", "5y"],
            value="1y",
            label="History"
        )

        load_button = gr.Button(
            "Load charts",
            variant="primary"
        )

    status = gr.Markdown(
        "Ready. Load a ticker to display its charts."
    )

    with gr.Row():

        with gr.Group():

            price_plot = build_price_panel(
                ticker_input,
                period_input,
                load_button
            )

        with gr.Group():

            metals_plot, metal_dropdown = build_metals_panel(
                demo,
                ticker_input,
                period_input,
                load_button
            )
    # Second row: New comparison and Achille's crypto panel
    with gr.Row():

        with gr.Group():

            gr.Markdown("### Investment Performance Comparison")

            vnq_plot = gr.Plot(
                value=initial_plot("Investment comparison"),
                label="Investment comparison"
            )

        # Achille's original crypto panel remains unchanged
        crypto_plot, crypto_input = build_crypto_panel(
            ticker_input,
            period_input,
            load_button
        )

    load_button.click(
        fn=update_dashboard,
        inputs=[ticker_input, period_input, metal_dropdown, crypto_input],
        outputs=[status, vnq_plot]
    )
    metal_dropdown.change(
        fn=update_dashboard,
        inputs=[ticker_input, period_input, metal_dropdown, crypto_input],
        outputs=[status, vnq_plot]
    )

    crypto_input.change(
        fn=update_dashboard,
        inputs=[ticker_input, period_input, metal_dropdown, crypto_input],
        outputs=[status, vnq_plot]
    )


demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
